# target speaker extraction demo

**use the `.venv` kernel from `uv sync`**

two examples using curriculum hard (our best model):
- **example 1** (`mix_librispeech_08338`): 100% overlap, +5 dB SIR — baseline 98% → extracted 16% WER. 11s clip, 9.0 dB interferer suppression.
- **example 2** (`mix_librispeech_00956`): 100% overlap, 0 dB SIR — baseline 93% → extracted 7% WER. steepest WER decline.

In [ ]:
import os, sys
from pathlib import Path

# find repo root by locating pyproject.toml, starting from cwd
p = Path(os.getcwd())
while p != p.parent:
    if (p / "pyproject.toml").exists():
        break
    p = p.parent
else:
    # fallback: hardcoded path
    p = Path.home() / "TargetTTS"

os.chdir(p)
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

import torch
import jiwer
import soundfile as sf
import IPython.display as ipd
from transformers import WhisperProcessor, WhisperForConditionalGeneration

from src.preprocess import preprocess_audio, normalize_text, TARGET_SAMPLE_RATE
from src.extraction_net import ExtractionNet, compute_stft, apply_mask_and_istft
from src.speaker_encoder import SpeakerEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ExtractionNet(input_power=0.3).to(device)
ckpt = torch.load("checkpoints/curriculum_hard/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

speaker_encoder = SpeakerEncoder(device=device)

whisper_proc = WhisperProcessor.from_pretrained("openai/whisper-tiny")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
whisper_model.eval()

def transcribe(audio_array):
    inputs = whisper_proc(audio_array, sampling_rate=TARGET_SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    forced_decoder_ids = whisper_proc.get_decoder_prompt_ids(language="english", task="transcribe")
    with torch.no_grad():
        ids = whisper_model.generate(inputs["input_features"], forced_decoder_ids=forced_decoder_ids)
    return normalize_text(whisper_proc.batch_decode(ids, skip_special_tokens=True)[0])

def extract(mix_path, enrollment_path):
    waveform = preprocess_audio(mix_path)
    magnitude, phase = compute_stft(waveform)
    emb = speaker_encoder.encode_file(enrollment_path).unsqueeze(0).to(device)
    with torch.no_grad():
        mask = model(magnitude.to(device), emb)
    return apply_mask_and_istft(magnitude.to(device), phase.to(device), mask).squeeze(0).cpu().numpy()

MIX_DIR = "data/synthetic_mixtures/librispeech_extraction"
print(f"loaded on {device}, cwd: {os.getcwd()}")

---
## example 1 — `mix_librispeech_08338`
100% overlap, +5 dB SIR — baseline 98% WER → extracted 16% WER. 11s clip, 9.0 dB interferer suppression.

In [ ]:
target_path  = "data/librispeech/LibriSpeech/test-clean/1320/122617/1320-122617-0032.flac"
enroll_path  = "data/librispeech/LibriSpeech/test-clean/1320/122617/1320-122617-0041.flac"
mix_path     = f"{MIX_DIR}/mix_librispeech_08338.wav"
reference    = normalize_text("keep silent as long as may be and it would be wise when you do speak to break out suddenly in one of your shoutings which will serve to remind the indians that you are not altogether as responsible as men should be")

target_audio = preprocess_audio(target_path).squeeze(0).numpy()
mix_audio, _ = sf.read(mix_path, dtype="float32")
clean_pred    = transcribe(target_audio)
baseline_pred = transcribe(mix_audio)

print("clean target:")
display(ipd.Audio(target_audio, rate=TARGET_SAMPLE_RATE))
print(f"reference:         {reference}")
print(f"whisper on clean:  {clean_pred}  (WER: {jiwer.wer(reference, clean_pred):.0%})")
print()
print("mixture (2 speakers, SIR=+5 dB, 100% overlap):")
display(ipd.Audio(mix_audio, rate=TARGET_SAMPLE_RATE))
print(f"baseline:          {baseline_pred}  (WER: {jiwer.wer(reference, baseline_pred):.0%})")

In [ ]:
enroll_audio    = preprocess_audio(enroll_path).squeeze(0).numpy()
extracted_audio = extract(mix_path, enroll_path)
extracted_pred  = transcribe(extracted_audio)
extracted_wer   = jiwer.wer(reference, extracted_pred)

print("enrollment clip (same speaker, different utterance):")
display(ipd.Audio(enroll_audio, rate=TARGET_SAMPLE_RATE))
print()
print("extracted audio:")
display(ipd.Audio(extracted_audio, rate=TARGET_SAMPLE_RATE))
print(f"extracted:  {extracted_pred}  (WER: {extracted_wer:.0%})")
print(f"reference:  {reference}")

---
## example 2 — `mix_librispeech_00956`
100% overlap, 0 dB SIR — baseline 93% WER → extracted 7% WER. steepest WER decline.

In [ ]:
target_path  = "data/librispeech/LibriSpeech/test-clean/1320/122612/1320-122612-0000.flac"
enroll_path  = "data/librispeech/LibriSpeech/test-clean/1320/122612/1320-122612-0016.flac"
mix_path     = f"{MIX_DIR}/mix_librispeech_00956.wav"
reference    = normalize_text("since the period of our tale the active spirit of the country has surrounded it with a belt of rich and thriving settlements though none but the hunter or the savage is ever known even now to penetrate its wild recesses")

target_audio = preprocess_audio(target_path).squeeze(0).numpy()
mix_audio, _ = sf.read(mix_path, dtype="float32")
clean_pred    = transcribe(target_audio)
baseline_pred = transcribe(mix_audio)

print("clean target:")
display(ipd.Audio(target_audio, rate=TARGET_SAMPLE_RATE))
print(f"reference:         {reference}")
print(f"whisper on clean:  {clean_pred}  (WER: {jiwer.wer(reference, clean_pred):.0%})")
print()
print("mixture (2 speakers at equal loudness, 100% overlap):")
display(ipd.Audio(mix_audio, rate=TARGET_SAMPLE_RATE))
print(f"baseline:          {baseline_pred}  (WER: {jiwer.wer(reference, baseline_pred):.0%})")

In [ ]:
enroll_audio    = preprocess_audio(enroll_path).squeeze(0).numpy()
extracted_audio = extract(mix_path, enroll_path)
extracted_pred  = transcribe(extracted_audio)
extracted_wer   = jiwer.wer(reference, extracted_pred)

print("enrollment clip (same speaker, different utterance):")
display(ipd.Audio(enroll_audio, rate=TARGET_SAMPLE_RATE))
print()
print("extracted audio:")
display(ipd.Audio(extracted_audio, rate=TARGET_SAMPLE_RATE))
print(f"extracted:  {extracted_pred}  (WER: {extracted_wer:.0%})")
print(f"reference:  {reference}")